# GEE Water Layer Agent Review Notebook

这个 notebook 只用于人工审核独立原型，不接入主系统。它会复用 SatGPT 根目录 `.env` 中的环境变量，动态抓取 GEE 官方 `water` 标签数据集，并根据问题判断是否需要上图。

In [1]:
# 首次测试时请先执行本单元
%pip install -r ./requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:

from google.auth.transport.requests import AuthorizedSession
from google.oauth2 import service_account
from dotenv import load_dotenv
import os
load_dotenv()

credentials = service_account.Credentials.from_service_account_file(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))
scoped_credentials = credentials.with_scopes(
    ['https://www.googleapis.com/auth/cloud-platform'])
session = AuthorizedSession(scoped_credentials)

d:\Conda\envs\floodagent\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [ ]:
import geemap
import ee
ee.Authenticate()
ee.Initialize(project='flood-agent')

In [ ]:
imageCollection = ee.ImageCollection("GLOBAL_FLOOD_DB/MODIS_EVENTS/V1")
image = imageCollection.filterMetadata('id', 'equals', 3977).first(); # 筛选代码
map_id = image.getMapId()
tile_url = map_id["tile_fetcher"].url_format

print(tile_url)

https://earthengine.googleapis.com/v1/projects/flood-agent/maps/8ac17bfb1b52a1799099a71d1d416f44-26c148aae7f07c626d2f8fb4fc2046df/tiles/{z}/{x}/{y}


In [44]:
import folium

# 先建底图（中心点可改成你的研究区）
m = folium.Map(
    location=[29.3, 112.9],   # 洞庭湖附近
    zoom_start=8,
    tiles="OpenStreetMap"
)

# 叠加你的 tile 图层
folium.TileLayer(
    tiles=tile_url,
    name="GEE Tile",
    attr="Google Earth Engine",
    overlay=True,
    control=True,
    opacity=0.8
).add_to(m)

folium.LayerControl().add_to(m)

m

In [8]:
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import display, Markdown

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import water_gee_agent
importlib.reload(water_gee_agent)
from water_gee_agent import review_question

In [9]:
question = "请帮我看看 2024 年洞庭湖周边的水体和洪水分布，最好直接上图。"
location_hint = None
start_date = None
end_date = None
force_refresh_catalog = False

result, review_map = review_question(
    question=question,
    location_hint=location_hint,
    start_date=start_date,
    end_date=end_date,
    force_refresh_catalog=force_refresh_catalog,
)

display(Markdown(f"## 问题\n{result.question}"))
display(Markdown(f"## 是否需要可视化\n`{result.plan.need_visualization}`\n\n{result.plan.reason}"))
display(Markdown(f"## 助手回复\n{result.plan.answer}"))
display(Markdown(f"## 位置/时间\n- location: `{result.plan.location_hint}`\n- start: `{result.plan.start_date}`\n- end: `{result.plan.end_date}`\n- used_llm: `{result.plan.used_llm}`"))
gee_auth = getattr(result, 'gee_auth', {}) or {}
display(Markdown(f"## GEE 认证\n- mode: `{gee_auth.get('mode')}`\n- project_id: `{gee_auth.get('project_id')}`\n- account: `{gee_auth.get('account')}`\n- credentials_path: `{gee_auth.get('credentials_path')}`\n- tile_proxy_base_url: `{gee_auth.get('tile_proxy_base_url')}`"))

## 问题
请帮我看看 2024 年洞庭湖周边的水体和洪水分布，最好直接上图。

## 是否需要可视化
`True`

用户希望查看洞庭湖周边的水体和洪水分布，涉及空间分布和变化，因此需要可视化图层。

## 助手回复
我将为您提供洞庭湖周边的水体和洪水分布图。

## 位置/时间
- location: `洞庭湖`
- start: `2024-01-01`
- end: `2024-12-31`
- used_llm: `True`

## GEE 认证
- mode: `service_account`
- project_id: `None`
- account: `flood-agent@flood-agent.iam.gserviceaccount.com`
- credentials_path: `D:\2025\.private-key.json`
- tile_proxy_base_url: `http://127.0.0.1:6538`

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 14619)
Traceback (most recent call last):
  File "c:\Users\admin\.codex\worktrees\9791\SatGPT-app\experiments\gee_water_layer_agent\water_gee_agent.py", line 118, in do_GET
    self.wfile.write(tile_bytes)
  File "d:\Conda\envs\floodagent\Lib\socketserver.py", line 840, in write
    self._sock.sendall(b)
ConnectionAbortedError: [WinError 10053] 你的主机中的软件中止了一个已建立的连接。

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "d:\Conda\envs\floodagent\Lib\socketserver.py", line 692, in process_request_thread
    self.finish_request(request, client_address)
  File "d:\Conda\envs\floodagent\Lib\socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "d:\Conda\envs\floodagent\Lib\socketserver.py", line 761, in __init__
    self.handle()
  File "d:\Conda\envs\floodagent\Lib\http

In [10]:
result

AgentResult(question='请帮我看看 2024 年洞庭湖周边的水体和洪水分布，最好直接上图。', plan=AgentPlan(need_visualization=True, reason='用户希望查看洞庭湖周边的水体和洪水分布，涉及空间分布和变化，因此需要可视化图层。', answer='我将为您提供洞庭湖周边的水体和洪水分布图。', location_hint='洞庭湖', start_date='2024-01-01', end_date='2024-12-31', selected_layers=[LayerDecision(slug='USGS_WBD_2017_HUC06', title='HUC06: USGS Watershed Boundary Dataset of Basins | Earth Engine Data Catalog | Google for Developers', asset_id='USGS/WBD/2017/HUC06', asset_type='FeatureCollection', why='该图层提供了流域边界数据，可以帮助理解水体的分布情况。', url='https://developers.google.com/earth-engine/datasets/catalog/USGS_WBD_2017_HUC06?hl=zh-cn'), LayerDecision(slug='WWF_HydroATLAS_v1_Basins_level03', title='WWF HydroATLAS Basins Level 03 | Earth Engine Data Catalog | Google for Developers', asset_id='WWF/HydroATLAS/v1/Basins/level03', asset_type='FeatureCollection', why='该图层包含高空间分辨率的流域水文环境属性信息，有助于分析水体分布。', url='https://developers.google.com/earth-engine/datasets/catalog/WWF_HydroATLAS_v1_Basins_level03?hl=zh-cn')], used_llm=

In [11]:
token_usage_df = pd.DataFrame([
    {
        "stage": item.stage,
        "prompt_tokens": item.prompt_tokens,
        "completion_tokens": item.completion_tokens,
        "total_tokens": item.total_tokens,
        "source": item.source,
        "note": item.note,
    }
    for item in result.token_usage
])
display(Markdown("## Token 使用明细"))
token_usage_df

## Token 使用明细

,stage,prompt_tokens,completion_tokens,total_tokens,source,note
0,catalog_lookup,0,0,0,none,官方 water 标签页抓取与本地缓存读取，不涉及 LLM token。
1,shortlist_ranking,0,0,0,none,本地关键词排序，不涉及 LLM token。
2,planner_llm,2568,216,2784,api_usage,来自兼容 Chat Completions 响应的 usage 字段。
3,map_render,0,0,0,none,GEE 地图渲染与瓦片请求不经过 LLM，不产生模型 token。


In [17]:
selected_df = pd.DataFrame([
    {
        "slug": layer.slug,
        "title": layer.title,
        "asset_id": layer.asset_id,
        "asset_type": layer.asset_type,
        "why": layer.why,
        "official_url": layer.url,
    }
    for layer in result.plan.selected_layers
])
selected_df

,slug,title,asset_id,asset_type,why,official_url
0,USGS_WBD_2017_HUC06,HUC06: USGS Watershed Boundary Dataset of Basi...,USGS/WBD/2017/HUC06,FeatureCollection,该图层提供了流域边界数据，可以帮助理解水体的分布情况。,https://developers.google.com/earth-engine/dat...
1,WWF_HydroATLAS_v1_Basins_level03,WWF HydroATLAS Basins Level 03 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level03,FeatureCollection,该图层包含高空间分辨率的流域水文环境属性信息，有助于分析水体分布。,https://developers.google.com/earth-engine/dat...


In [18]:
rendered_df = pd.DataFrame(result.rendered_layers)
rendered_df

,slug,title,asset_id,asset_type,tile_url,earth_engine_tile_url,sample_zoom,sample_x,sample_y,sample_browser_tile_url,sample_earth_engine_tile_url,vis_meta,tile_loading_mode,official_url
0,USGS_WBD_2017_HUC06,HUC06: USGS Watershed Boundary Dataset of Basi...,USGS/WBD/2017/HUC06,FeatureCollection,http://127.0.0.1:6538/ee-tiles/usgs_wbd_2017_h...,https://earthengine.googleapis.com/v1/projects...,6,51,26,http://127.0.0.1:6538/ee-tiles/usgs_wbd_2017_h...,https://earthengine.googleapis.com/v1/projects...,{'style': 'feature-outline'},authenticated_proxy_tiles,https://developers.google.com/earth-engine/dat...
1,WWF_HydroATLAS_v1_Basins_level03,WWF HydroATLAS Basins Level 03 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level03,FeatureCollection,http://127.0.0.1:6538/ee-tiles/wwf_hydroatlas_...,https://earthengine.googleapis.com/v1/projects...,6,51,26,http://127.0.0.1:6538/ee-tiles/wwf_hydroatlas_...,https://earthengine.googleapis.com/v1/projects...,{'style': 'feature-outline'},authenticated_proxy_tiles,https://developers.google.com/earth-engine/dat...


In [19]:
display(Markdown("## Raw Tile URLs"))
if rendered_df.empty:
    print('No rendered layers.')
else:
    for idx, row in rendered_df.iterrows():
        print(f"[{idx}] {row.get('title', '')}")
        print('browser_tile_url_template:')
        print(row.get('tile_url', 'NO_TILE_URL'))
        print('earth_engine_tile_url_template:')
        print(row.get('earth_engine_tile_url', 'NO_EARTH_ENGINE_TILE_URL'))
        print('sample_browser_tile_url:')
        print(row.get('sample_browser_tile_url', 'NO_SAMPLE_BROWSER_TILE_URL'))
        print('sample_earth_engine_tile_url:')
        print(row.get('sample_earth_engine_tile_url', 'NO_SAMPLE_EARTH_ENGINE_TILE_URL'))
        print('-' * 120)

## Raw Tile URLs

[0] HUC06: USGS Watershed Boundary Dataset of Basins | Earth Engine Data Catalog | Google for Developers
browser_tile_url_template:
http://127.0.0.1:6538/ee-tiles/usgs_wbd_2017_huc06/{z}/{x}/{y}
earth_engine_tile_url_template:
https://earthengine.googleapis.com/v1/projects/earthengine-legacy/maps/990f74913589669468a354c55db85b04-3cb367eb2dda1b0c6ddf9bfd4064f416/tiles/{z}/{x}/{y}
sample_browser_tile_url:
http://127.0.0.1:6538/ee-tiles/usgs_wbd_2017_huc06/6/51/26
sample_earth_engine_tile_url:
https://earthengine.googleapis.com/v1/projects/earthengine-legacy/maps/990f74913589669468a354c55db85b04-3cb367eb2dda1b0c6ddf9bfd4064f416/tiles/6/51/26
------------------------------------------------------------------------------------------------------------------------
[1] WWF HydroATLAS Basins Level 03 | Earth Engine Data Catalog | Google for Developers
browser_tile_url_template:
http://127.0.0.1:6538/ee-tiles/wwf_hydroatlas_v1_basins_level03/{z}/{x}/{y}
earth_engine_tile_url_template:
https://ea

In [20]:
display(Markdown("## 样例 Tile 请求测试"))
if rendered_df.empty:
    print('No rendered layers to test.')
else:
    import requests
    test_url = rendered_df.iloc[0].get('sample_browser_tile_url')
    print('GET', test_url)
    resp = requests.get(test_url, timeout=30)
    print('status=', resp.status_code)
    print('content_type=', resp.headers.get('Content-Type'))
    print('bytes=', len(resp.content))

## 样例 Tile 请求测试

GET http://127.0.0.1:6538/ee-tiles/usgs_wbd_2017_huc06/6/51/26
status= 200
content_type= image/png
bytes= 334


In [21]:
if result.map_ready and review_map is not None:
    display(Markdown("## 地图预览"))
    display(review_map)
else:
    display(Markdown(f"## 地图未生成\n{result.map_error or '本次判断无需可视化。'}"))

## 地图预览

In [22]:
if result.map_ready and review_map is not None:
    html_path = NOTEBOOK_DIR / 'last_review_map.html'
    review_map.save(str(html_path))
    display(Markdown(f"地图 HTML 已导出到: `{html_path}`"))

地图 HTML 已导出到: `c:\Users\admin\.codex\worktrees\9791\SatGPT-app\experiments\gee_water_layer_agent\last_review_map.html`

In [23]:
shortlist_df = pd.DataFrame([
    {
        "slug": item.slug,
        "title": item.title,
        "asset_id": item.asset_id,
        "asset_type": item.asset_type,
        "official_url": item.url,
    }
    for item in result.shortlist
])
shortlist_df

,slug,title,asset_id,asset_type,official_url
0,HYCOM_sea_temp_salinity,"HYCOM: Hybrid Coordinate Ocean Model, Water Te...",HYCOM/sea_temp_salinity,ImageCollection,https://developers.google.com/earth-engine/dat...
1,HYCOM_sea_water_velocity,"HYCOM: Hybrid Coordinate Ocean Model, Water Ve...",HYCOM/sea_water_velocity,ImageCollection,https://developers.google.com/earth-engine/dat...
2,USGS_WBD_2017_HUC06,HUC06: USGS Watershed Boundary Dataset of Basi...,USGS/WBD/2017/HUC06,FeatureCollection,https://developers.google.com/earth-engine/dat...
3,USGS_WBD_2017_HUC08,HUC08: USGS Watershed Boundary Dataset of Subb...,USGS/WBD/2017/HUC08,FeatureCollection,https://developers.google.com/earth-engine/dat...
4,HYCOM_sea_surface_elevation,"HYCOM: Hybrid Coordinate Ocean Model, Sea Surf...",HYCOM/sea_surface_elevation,ImageCollection,https://developers.google.com/earth-engine/dat...
5,WWF_HydroATLAS_v1_Basins_level03,WWF HydroATLAS Basins Level 03 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level03,FeatureCollection,https://developers.google.com/earth-engine/dat...
6,WWF_HydroATLAS_v1_Basins_level04,WWF HydroATLAS Basins Level 04 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level04,FeatureCollection,https://developers.google.com/earth-engine/dat...
7,WWF_HydroATLAS_v1_Basins_level05,WWF HydroATLAS Basins Level 05 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level05,FeatureCollection,https://developers.google.com/earth-engine/dat...
8,WWF_HydroATLAS_v1_Basins_level06,WWF HydroATLAS Basins Level 06 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level06,FeatureCollection,https://developers.google.com/earth-engine/dat...
9,WWF_HydroATLAS_v1_Basins_level07,WWF HydroATLAS Basins Level 07 | Earth Engine ...,WWF/HydroATLAS/v1/Basins/level07,FeatureCollection,https://developers.google.com/earth-engine/dat...
